# Large Document → Knowledge Graph (Chunking + Batch)

**The question everyone asks:** *"What happens with a 500-page document? You can't send it all to the LLM!"*

**Answer:** You chunk it, extract from each chunk separately, then **merge** entities across chunks into one unified graph.

```
500-page document
       │
       ▼
  Split into 50 chunks
       │
       ├── Chunk 1 → LLM → entities: [Einstein, Princeton]
       ├── Chunk 2 → LLM → entities: [Einstein, Relativity]
       ├── Chunk 3 → LLM → entities: [Princeton, New Jersey]
       │              ...
       ▼
  MERGE all (Einstein appears 10 times → 1 node)
       │
       ▼
  One connected Knowledge Graph
```

The **magic**: entities from Chunk 1 connect to entities from Chunk 47 because they share a node.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv(Path(".").resolve().parent / ".env")
load_dotenv(Path(".").resolve() / ".env")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
NEO4J_URI = "bolt://localhost:7687"
NEO4J_AUTH = ("neo4j", "workshop2024")

print("Setup complete!")

---
## Step 1: Create a "large" document

We'll simulate a large document by combining multiple sections. In reality, this could be a PDF, a book, or hundreds of pages.

In [ ]:
# Simulate a large document with multiple sections
# Each section mentions overlapping entities — this is what creates cross-chunk connections

large_document = """
SECTION 1: FOUNDING AND EARLY HISTORY

TechVentures Inc. was founded in 2018 by Dr. Maria Santos and Raj Patel in Boston, Massachusetts.
Maria had previously been a professor at MIT specializing in natural language processing, while
Raj was a serial entrepreneur who had sold his previous company DataFlow to Microsoft for $200M.
The founding team also included Elena Volkov as Chief Scientist, who came from IBM Research where
she had led the Watson NLP team for six years. Their initial funding of $5M came from Founders Fund
and Khosla Ventures. The company's first office was in Cambridge, near the MIT campus.

SECTION 2: PRODUCT DEVELOPMENT

TechVentures launched its first product, TextAnalyzer, in 2019. TextAnalyzer used transformer-based
models to automatically extract insights from legal documents. The product was built by Elena Volkov's
research team using PyTorch and deployed on AWS. Their first major customer was Baker McKenzie,
one of the world's largest law firms. By 2020, they had also signed Clifford Chance and Allen & Overy.
Maria Santos personally led the sales effort to these law firms, leveraging her academic reputation.

SECTION 3: GROWTH AND EXPANSION

In 2021, TechVentures raised a $50M Series B led by Andreessen Horowitz with participation from
existing investors Founders Fund and Khosla Ventures. The company opened a new office in San Francisco
and hired Tom Williams as VP of Engineering from Google, where he had managed the BERT team. Tom
brought five senior engineers with him from Google. Raj Patel transitioned from CEO to Chairman,
and the board hired Sarah Mitchell from McKinsey as the new CEO.

SECTION 4: NEW PRODUCTS AND PARTNERSHIPS

Under Sarah Mitchell's leadership, TechVentures launched two new products in 2022. ContractAI
automated contract review and was built in partnership with DocuSign. KnowledgeHub was an internal
knowledge management tool that used graph databases, developed in collaboration with Neo4j.
Elena Volkov led the research for KnowledgeHub, combining her NLP expertise with graph technology.
Tom Williams' engineering team built both products using a microservices architecture on Kubernetes.

SECTION 5: COMPETITION AND CHALLENGES

TechVentures faced increasing competition from OpenAI's GPT-based products and Google's Document AI.
In response, Elena Volkov's team pivoted to using retrieval-augmented generation (RAG), integrating
Anthropic's Claude API alongside their existing models. The company also faced a lawsuit from a
competitor, LegalTech Solutions, over patent infringement related to document extraction methods.
Raj Patel, as Chairman, led the legal defense strategy. The case was settled out of court in 2023.

SECTION 6: INTERNATIONAL EXPANSION

In 2023, TechVentures expanded to Europe, opening offices in London and Berlin. They hired
Dr. Hans Mueller from SAP as Head of European Operations. Hans had spent 15 years at SAP leading
their enterprise AI division. The European expansion was funded by a $100M Series C led by SoftBank
Vision Fund, with Andreessen Horowitz participating again. Key European customers included Deutsche
Bank, Siemens, and the UK Government's Cabinet Office.

SECTION 7: CURRENT STATE AND FUTURE

As of 2025, TechVentures has 500 employees across Boston (HQ), San Francisco, London, and Berlin.
Sarah Mitchell remains CEO and has been featured in Fortune's "40 Under 40" list. Maria Santos
returned to MIT as an adjunct professor but remains on the board. Elena Volkov was promoted to
CTO after Tom Williams left to join Anthropic as VP of Engineering. Raj Patel serves on the boards
of three other AI startups. The company is rumored to be preparing for an IPO in late 2026,
with Goldman Sachs and Morgan Stanley as underwriters.
"""

print(f"Document size: {len(large_document)} characters")
print(f"That's about {len(large_document.split())} words")
print(f"\nSections: {large_document.count('SECTION')}")

---
## Step 2: Chunk the document

Split into chunks with **overlap** so entities near chunk boundaries appear in multiple chunks.

In [ ]:
def chunk_text(text, chunk_size=1500, overlap=300):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap  # overlap ensures entities near boundaries appear in 2 chunks
    return chunks

chunks = chunk_text(large_document, chunk_size=1500, overlap=300)

print(f"Split into {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {len(chunk)} chars | Preview: {chunk[:80].strip()}...")

---
## Step 3: Extract from each chunk separately

We send each chunk to the LLM independently. The key: **the same entity (e.g., "Elena Volkov") appears in multiple chunks** — we'll merge them later.

In [ ]:
class Entity(BaseModel):
    name: str = Field(description="Entity name")
    type: str = Field(description="PERSON, COMPANY, PRODUCT, CITY, INVESTOR, UNIVERSITY, TECHNOLOGY")
    description: str = Field(description="One-line description")

class Relationship(BaseModel):
    source: str = Field(description="Source entity name")
    target: str = Field(description="Target entity name")
    type: str = Field(description="Relationship type")
    description: str = Field(description="One-line description")

class ChunkKG(BaseModel):
    entities: list[Entity] = Field(default_factory=list)
    relationships: list[Relationship] = Field(default_factory=list)

structured_llm = llm.with_structured_output(ChunkKG)

# Extract from each chunk
all_entities = []
all_relationships = []

for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i+1}/{len(chunks)}...")
    
    kg = structured_llm.invoke([
        SystemMessage(content="""Extract entities and relationships from this text chunk.
Types: PERSON, COMPANY, PRODUCT, CITY, INVESTOR, UNIVERSITY, TECHNOLOGY
Only extract what is explicitly stated."""),
        HumanMessage(content=f"Extract from this chunk:\n\n{chunk}"),
    ])
    
    for e in kg.entities:
        e_dict = e.model_dump()
        e_dict["source_chunk"] = i + 1
        all_entities.append(e_dict)
    
    for r in kg.relationships:
        r_dict = r.model_dump()
        r_dict["source_chunk"] = i + 1
        all_relationships.append(r_dict)
    
    print(f"  → {len(kg.entities)} entities, {len(kg.relationships)} relationships")

print(f"\nTotal (before merge): {len(all_entities)} entities, {len(all_relationships)} relationships")

---
## Step 4: Merge duplicate entities

"Elena Volkov" appears in chunks 1, 2, 4, 5, and 7. We merge them into **one node** with the richest description.

In [ ]:
# Merge entities by name (case-insensitive)
merged = {}
for e in all_entities:
    key = e["name"].strip().lower()
    if key in merged:
        existing = merged[key]
        # Keep the longer description
        if len(e["description"]) > len(existing["description"]):
            existing["description"] = e["description"]
        existing["chunks"].append(e["source_chunk"])
    else:
        merged[key] = {
            "name": e["name"],
            "type": e["type"],
            "description": e["description"],
            "chunks": [e["source_chunk"]],
        }

unique_entities = list(merged.values())

print(f"Before merge: {len(all_entities)} entities")
print(f"After merge:  {len(unique_entities)} unique entities\n")

# Show entities that appeared in multiple chunks — these are the cross-chunk connections!
print("Entities appearing in multiple chunks (these create cross-chunk connections):")
for e in sorted(unique_entities, key=lambda x: len(x['chunks']), reverse=True):
    if len(e['chunks']) > 1:
        print(f"  {e['name']:<25} appeared in chunks {e['chunks']}")

---
## Step 5: Load into Neo4j with MERGE (not CREATE)

**MERGE** = create if doesn't exist, match if it does. This prevents duplicates even if we process chunks in parallel.

In [ ]:
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)

with driver.session() as s:
    # Delete everything first
    s.run("MATCH (n) DETACH DELETE n")
    print("Cleared Neo4j\n")
    
    # Load merged entities (MERGE avoids duplicates)
    for e in unique_entities:
        s.run("""MERGE (n:Entity {name: $name})
               SET n.type = $type, n.description = $desc, n.chunks = $chunks""",
              name=e["name"], type=e["type"], desc=e["description"],
              chunks=str(e["chunks"]))
    
    # Load relationships (MERGE avoids duplicate edges too)
    loaded = 0
    for r in all_relationships:
        result = s.run(
            """MATCH (a:Entity {name: $src}), (b:Entity {name: $tgt})
               MERGE (a)-[rel:RELATES_TO {type: $type}]->(b)
               SET rel.description = $desc
               RETURN count(*) AS c""",
            src=r["source"], tgt=r["target"], type=r["type"], desc=r["description"])
        if result.single()["c"] > 0:
            loaded += 1
    
    nodes = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
    edges = s.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]

print(f"Knowledge Graph: {nodes} nodes, {edges} relationships")
print(f"Loaded {loaded} relationships out of {len(all_relationships)} total (rest were duplicates or missing nodes)")

driver.close()

### Visualize

In [ ]:
from pyvis.network import Network
import os

COLORS = {"PERSON": "#FF6B6B", "COMPANY": "#4ECDC4", "PRODUCT": "#45B7D1",
          "CITY": "#FFEAA7", "INVESTOR": "#F39C12", "UNIVERSITY": "#96CEB4",
          "TECHNOLOGY": "#DDA0DD"}

net = Network(height="800px", width="100%", directed=True, bgcolor="#1a1a2e",
              font_color="white", cdn_resources="remote")
net.barnes_hut(gravity=-5000, spring_length=200)

driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
with driver.session() as s:
    degrees = {}
    for r in s.run("MATCH (n) OPTIONAL MATCH (n)-[r]-() RETURN n.name AS name, count(r) AS deg"):
        degrees[r["name"]] = r["deg"]
    
    for r in s.run("MATCH (n:Entity) RETURN n.name AS name, n.type AS type, n.description AS desc, n.chunks AS chunks"):
        color = COLORS.get(r["type"], "#DFE6E9")
        size = 12 + degrees.get(r["name"], 0) * 4
        title = f"<b>{r['name']}</b><br>Type: {r['type']}<br>{r['desc']}<br>Chunks: {r['chunks']}"
        net.add_node(r["name"], label=r["name"], color=color, size=size, title=title)
    
    for r in s.run("MATCH (a)-[r]->(b) RETURN a.name AS s, b.name AS t, r.type AS type"):
        net.add_edge(r["s"], r["t"], label=r["type"], color="#636e72",
                     font={"size": 8, "color": "#b2bec3"})
driver.close()

html_path = os.path.abspath("large_doc_kg.html")
net.save_graph(html_path)
os.system(f"open '{html_path}'")
print(f"Hover over nodes to see which chunks they came from!")

---
## Step 6: Query — Cross-Chunk Questions

These questions require information from **different chunks** — exactly where classic RAG fails.

In [ ]:
def ask_llm(prompt, system="You are a helpful assistant."):
    return llm.invoke([SystemMessage(content=system), HumanMessage(content=prompt)]).content

def graph_rag(question):
    driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
    with driver.session() as s:
        names = [r["name"] for r in s.run("MATCH (n:Entity) RETURN n.name AS name")]
        entity_types = [r["t"] for r in s.run("MATCH (n:Entity) RETURN DISTINCT n.type AS t")]
        rel_types = [r["t"] for r in s.run("MATCH ()-[r]->() RETURN DISTINCT r.type AS t")]
    
    cypher = ask_llm(
        prompt=f"Convert to Cypher:\n\n{question}",
        system=f"""Neo4j expert. Schema:
Nodes: :Entity (name, type, description)
Entity types: {entity_types}
Relationships: :RELATES_TO (type, description)
Relationship types: {rel_types}
Entity names: {names}
Use toLower() and CONTAINS. Return ONLY Cypher."""
    ).strip().replace("```cypher", "").replace("```", "").strip()
    
    print(f"Cypher: {cypher}\n")
    
    try:
        with driver.session() as s:
            rows = [dict(r) for r in s.run(cypher)]
    except:
        rows = []
    if not rows:
        with driver.session() as s:
            rows = [dict(r) for r in s.run(
                "MATCH (a:Entity)-[r]->(b:Entity) RETURN a.name AS source, r.type AS rel, b.name AS target"
            )]
    
    driver.close()
    context = "\n".join([str(r) for r in rows])
    return ask_llm(
        f"Graph results:\n{context}\n\nQuestion: {question}",
        system="Answer using ONLY graph results. Cite relationships."
    )

In [ ]:
# This answer requires info from Section 1 + Section 7 (different chunks!)
print(graph_rag("Trace Elena Volkov's career journey from IBM to CTO of TechVentures."))

In [ ]:
# This spans Section 1, 3, and 6 (three different chunks)
print(graph_rag("List all investors across all funding rounds and how much each round raised."))

In [ ]:
# This requires connecting people who appeared in completely different sections
print(graph_rag("Which employees left TechVentures and where did they go?"))

---
## Why This Works

```
Chunk 1: "Elena Volkov came from IBM Research"     → (Elena) --[PREVIOUSLY_AT]--> (IBM)
Chunk 2: "Elena Volkov's research team built..."   → (Elena) --[LEADS]--> (Research Team)
Chunk 4: "Elena Volkov led research for KnowledgeHub" → (Elena) --[BUILT]--> (KnowledgeHub)
Chunk 7: "Elena Volkov was promoted to CTO"        → (Elena) --[CTO_OF]--> (TechVentures)
```

After merge, ONE "Elena Volkov" node connects to IBM, Research Team, KnowledgeHub, and TechVentures.

**Classic RAG** would return one of these chunks. **Graph RAG** returns the complete picture.

### Scaling to truly large documents

For 500+ page documents:
1. **Chunk size**: 1500-2000 chars with 300 char overlap
2. **Parallel extraction**: Process chunks concurrently with `asyncio`
3. **Batch loading**: Use Neo4j's `UNWIND` for bulk inserts instead of one-by-one
4. **Entity resolution**: Use LLM or embedding similarity to merge "Dr. Elena Volkov" with "Elena" with "Volkov"

The pattern is always the same: **chunk → extract → merge → load → query**.